# Finale Projektdokumentation
## Vorhersage von Gehaltsklassen in Stellenausschreibungen

### Modulkontext
Semesterarbeit im Data-Science-Modul

### Projektziel
Ziel des Projekts war es, zu untersuchen, ob sich aus Merkmalen von Stellenanzeigen ein Gehaltsniveau mit Methoden des Machine Learning vorhersagen lässt. Dabei stand nicht die exakte Vorhersage eines individuellen Gehalts im Vordergrund, sondern die Einordnung in die drei Klassen **niedrig**, **mittel** und **hoch**.

Da die verwendeten Stellenanzeigen in der Regel selbst keine direkten Gehaltsangaben enthalten, musste im Hauptsetup zunächst eine Proxy-Zielvariable konstruiert werden. Im späteren Projektverlauf wurde zusätzlich eine alternative Datenquelle mit direkteren Gehaltsfeldern untersucht, um die Grenzen und Potenziale des ursprünglichen Ansatzes kritisch einzuordnen.

## Problemstellung und Motivation

Stellenanzeigen enthalten häufig umfangreiche Informationen zu Tätigkeit, Anforderungen, Arbeitsort und Arbeitgeber, jedoch fehlt oft eine zentrale Information: das Gehalt. Für Bewerbende erschwert dies die Vergleichbarkeit verschiedener Angebote. Auch für Plattformen oder arbeitsmarktbezogene Analysen ist das Fehlen strukturierter Gehaltsinformationen problematisch.

Die Grundidee des Projekts bestand daher darin, aus vorhandenen Merkmalen einer Stellenanzeige ein Gehaltsniveau vorherzusagen. Das Projekt ist damit sowohl aus Nutzersicht als auch aus analytischer Sicht relevant: Es könnte genutzt werden, um unvollständige Datensätze anzureichern oder um Stellenanzeigen grob nach Gehaltsniveau zu klassifizieren.

## Datenbasis und methodischer Grundansatz

Im Hauptteil des Projekts wurden Stellenanzeigen aus der Jobsuche-API der Bundesagentur für Arbeit verwendet. Diese Daten enthielten unter anderem die Merkmale:

- `titel`
- `beruf`
- `ort`
- `region`

Da die BA-Stellenanzeigen meist kein explizites Gehalt enthalten, wurde im Projekt eine Proxy-Zielvariable konstruiert. Dafür wurden die Stellenanzeigen mit externen Entgeltdaten der Bundesagentur für Arbeit auf Ebene von Berufsgruppe und Region verknüpft. Auf Grundlage der zugeordneten Medianentgelte wurde anschließend mit `qcut` eine Klassifikation in drei Gehaltsklassen erzeugt: **niedrig**, **mittel** und **hoch**.

Im späteren Projektverlauf wurde zusätzlich die Adzuna-API untersucht, da diese teilweise direkte Gehaltsfelder wie `salary_min` und `salary_max` bereitstellt.

## Vorgehensweise und Sprint-Struktur

Das Projekt wurde iterativ aufgebaut und orientierte sich an einer CRISP-DM-ähnlichen Struktur.

### Sprint 1
Datenakquise über die API der Bundesagentur für Arbeit, erste Bereinigung und Exploration der Rohdaten.

### Sprint 2
Verknüpfung mit externen Entgeltdaten, KldB-basierte Berufsgruppenzuordnung und Konstruktion der Zielvariable `gehaltsklasse`.

### Sprint 3
Modellierung, Evaluation, Cross-Validation, Ablation und Fehleranalyse auf dem BA-Setup.

### Sprint 4
Prüfung einer alternativen Datenquelle mit direkteren Gehaltsfeldern über die Adzuna-API.

### Sprint 5
Vergleich des BA- und Adzuna-Ansatzes sowie ein kleiner Machbarkeitstest mit einem direkten Adzuna-Gehaltssetup.

## Sprint 1: Datenakquise und Exploration

In Sprint 1 wurden Stellenanzeigen aus der Jobsuche-API der Bundesagentur für Arbeit geladen und als Ausgangsdatensatz aufbereitet. Der Fokus lag zunächst auf dem Datenverständnis:

- Welche Felder sind vorhanden?
- Welche Merkmale sind für das spätere Problem potenziell relevant?
- Wie viele fehlende Werte und Duplikate treten auf?
- Welche geographischen Einschränkungen sind notwendig?

Ein zentrales Ergebnis aus diesem Sprint war, dass insbesondere `titel`, `beruf`, `ort` und `region` als relevante Merkmale für die spätere Modellierung geeignet sind. Gleichzeitig wurde deutlich, dass direkte Gehaltsangaben in dieser Datenquelle nicht zuverlässig vorhanden sind. Daraus ergab sich die Konsequenz für den nächsten Sprint: Die Zielvariable musste extern konstruiert werden.

## Sprint 2: Konstruktion der Proxy-Zielvariable

In Sprint 2 wurde die zentrale methodische Grundlage des Hauptprojekts geschaffen. Die Stellenanzeigen wurden über KldB-Codes auf 3-stellige Berufsgruppen abgebildet und anschließend mit externen BA-Entgeltdaten auf Ebene von Berufsgruppe und Region verknüpft.

Dadurch konnte jeder Stelle ein Medianentgelt zugeordnet werden. Dieses wurde anschließend nicht direkt als numerischer Regressionswert verwendet, sondern in drei Gehaltsklassen eingeteilt:

- niedrig
- mittel
- hoch

Diese Entscheidung war bewusst getroffen, da die zugrunde liegenden Entgeltdaten bereits eine aggregierte und nicht individuelle Größe darstellen. Eine Klassifikation erschien deshalb methodisch robuster als eine Regression auf exakte Euro-Beträge.

Die Konsequenz für Sprint 3 war, dass nun ein klar definiertes überwachtes Lernproblem vorlag.

## Sprint 3: Modellierung und Evaluation des BA-Setups

In Sprint 3 wurde das eigentliche Machine-Learning-Setup auf Basis des BA-Datensatzes aufgebaut. Verwendet wurden einfache, aber nachvollziehbare Merkmale aus den Stellenanzeigen, insbesondere `titel`, `beruf`, `ort` und `region`. Diese wurden zu einem gemeinsamen Textfeld kombiniert und anschließend mit TF-IDF in numerische Features umgewandelt.

Getestet wurden drei Modelle:

- Baseline mit DummyClassifier
- Logistic Regression
- Random Forest

Die Ergebnisse zeigten, dass beide echten Modelle die Baseline deutlich übertrafen. Besonders der Random Forest erzielte eine hohe Vorhersageleistung. Gleichzeitig wurde die Evaluation durch Cross-Validation abgesichert.

In [44]:
import pandas as pd

ba_results = pd.DataFrame({
    "Modell": ["Baseline", "Logistic Regression", "Random Forest"],
    "Accuracy": [0.337, 0.865, 0.891]
})

ba_results

,Modell,Accuracy
0,Baseline,0.337
1,Logistic Regression,0.865
2,Random Forest,0.891


Die Ergebnisse zeigen, dass sich die Proxy-Gehaltsklasse aus den verwendeten Merkmalen deutlich besser als durch eine triviale Vorhersage klassifizieren lässt. Allerdings musste früh reflektiert werden, dass eine hohe Accuracy hier nicht automatisch bedeutet, dass ein echtes individuelles Gehalt präzise vorhergesagt wird.

## Vertiefung in Sprint 3: Ablation und Fehleranalyse

Zur besseren Einordnung der Modellleistung wurde Sprint 3 um zusätzliche Analysen ergänzt. Besonders wichtig war die Ablation Study, in der verschiedene Merkmalskombinationen getestet wurden.

Das zentrale Ergebnis war, dass die Kombination aus **`beruf` und `region`** die stärkste Vorhersageleistung erzielte. Damit wurde empirisch bestätigt, dass ein großer Teil der Modellgüte aus genau den strukturellen Merkmalen stammt, die auch in die Konstruktion der Zielvariable eingeflossen sind.

Zusätzlich wurde eine Fehleranalyse durchgeführt. Dabei zeigte sich, dass Fehlklassifikationen vor allem zwischen benachbarten Klassen auftraten, also zum Beispiel zwischen `niedrig` und `mittel` oder zwischen `mittel` und `hoch`. Das spricht dafür, dass das Modell eher Unsicherheiten an Übergängen zeigt als völlig unplausible Vorhersagen.

In [45]:
ablation_results = pd.DataFrame({
    "Feature_Set": [
        "beruf_region",
        "titel_beruf_region",
        "alle_features",
        "titel_region",
        "titel_beruf",
        "nur_beruf",
        "nur_titel",
        "nur_ort",
        "nur_region"
    ],
    "CV_Mean_Accuracy": [0.904, 0.882, 0.853, 0.830, 0.780, 0.775, 0.714, 0.531, 0.499]
})

ablation_results

,Feature_Set,CV_Mean_Accuracy
0,beruf_region,0.904
1,titel_beruf_region,0.882
2,alle_features,0.853
3,titel_region,0.830
4,titel_beruf,0.780
5,nur_beruf,0.775
6,nur_titel,0.714
7,nur_ort,0.531
8,nur_region,0.499


Diese Analyse war für die methodische Reflexion besonders wichtig. Sie zeigt, dass das Modell im Hauptsetup nicht unabhängig ein Gehaltsmuster „entdeckt“, sondern stark die Struktur des Proxy-Labels rekonstruiert. Genau darin liegt sowohl die Stärke als auch die zentrale Einschränkung des BA-Ansatzes.

## Sprint 4: Erweiterung durch Adzuna

Um die wichtigste methodische Schwäche des BA-Setups aktiv zu prüfen, wurde in Sprint 4 eine alternative Datenquelle mit direkteren Gehaltsfeldern untersucht: die Adzuna-API.

Diese Datenquelle stellte theoretisch eine attraktivere Grundlage dar, da dort teilweise direkte Felder wie `salary_min` und `salary_max` verfügbar waren. In der praktischen Nutzung zeigte sich jedoch schnell, dass die Datenqualität deutlich problematischer war:

- viele fehlende Gehaltsfelder
- zahlreiche Duplikate
- auffällige Ausreißer und unplausible Werte

Nach Dublettenbereinigung und Plausibilitätsfilterung blieb nur ein relativ kleiner Datensatz übrig. Damit wurde deutlich, dass eine Datenquelle mit direkteren Gehaltsfeldern nicht automatisch die bessere Arbeitsgrundlage sein muss.

In [46]:
adzuna_summary = pd.DataFrame({
    "Kennzahl": [
        "Anzeigen nach Dublettenbereinigung",
        "Anzeigen mit salary_min und salary_max",
        "Anzeigen nach Plausibilitätsbereinigung",
        "Anteil nach Bereinigung"
    ],
    "Wert": [15391, 813, 637, 0.041]
})

adzuna_summary

,Kennzahl,Wert
0,Anzeigen nach Dublettenbereinigung,15391.000
1,Anzeigen mit salary_min und salary_max,813.000
2,Anzeigen nach Plausibilitätsbereinigung,637.000
3,Anteil nach Bereinigung,0.041


Die Konsequenz aus Sprint 4 war daher nicht, das BA-Setup zu verwerfen, sondern den Adzuna-Ansatz als methodisch interessante, aber praktisch eingeschränkte Ergänzung zu betrachten.

## Sprint 5: Vergleich der Datenstrategien und Machbarkeitstest

In Sprint 5 wurden die beiden Datenstrategien systematisch miteinander verglichen:

- BA-Setup mit Proxy-Gehaltsklasse
- Adzuna-Setup mit direkteren Gehaltsfeldern

Zusätzlich wurde auf dem bereinigten Adzuna-Datensatz ein kleines Testmodell trainiert, um zu prüfen, ob sich grundsätzlich ein sinnvolles Klassifikationssetup mit direkteren Gehaltsdaten aufbauen lässt.

In [47]:
adzuna_model_results = pd.DataFrame({
    "Modell": ["Baseline", "Logistic Regression", "Random Forest"],
    "Accuracy": [0.344, 0.594, 0.555]
})

adzuna_model_results

,Modell,Accuracy
0,Baseline,0.344
1,Logistic Regression,0.594
2,Random Forest,0.555


Die Ergebnisse zeigen, dass auch mit dem kleinen Adzuna-Datensatz ein sinnvolles Modell trainiert werden kann. Gleichzeitig blieb die Datenbasis jedoch deutlich kleiner und instabiler als im BA-Setup.

Der systematische Vergleich führte deshalb zu einer klaren Schlussentscheidung:

- Das **BA-Setup** bleibt für diese Arbeit der tragfähigere Hauptansatz, weil es eine größere und stabilere Datengrundlage bietet.
- Das **Adzuna-Setup** ist methodisch näher am eigentlichen Problem, eignet sich hier aber eher als ergänzende Machbarkeitsanalyse.

## Zentrale Ergebnisse des Gesamtprojekts

Die wichtigsten Ergebnisse des Projekts lassen sich wie folgt zusammenfassen:

1. Aus Stellenanzeigen lässt sich ein Gehaltsniveau grundsätzlich mit Machine Learning klassifizieren.
2. Im BA-Setup ist dafür jedoch eine Proxy-Zielvariable notwendig.
3. Die hohe Modellleistung im Hauptsetup wird stark durch berufs- und regionsbezogene Informationen getragen.
4. Die Ablation Study bestätigt, dass genau diese Merkmale den größten Teil der Vorhersagekraft liefern.
5. Eine alternative Datenquelle mit direkteren Gehaltsfeldern ist methodisch attraktiver, war in diesem Projekt aber praktisch zu lückenhaft, um den Hauptansatz zu ersetzen.

## Reflexion

Die größte Stärke des Projekts liegt in der iterativen Vorgehensweise. Die Arbeit blieb nicht bei einem ersten funktionierenden Setup stehen, sondern hat die zentrale methodische Schwäche aktiv weiterverfolgt und mit einer alternativen Datenquelle überprüft.

Die größte methodische Einschränkung liegt in der Zielvariable des BA-Setups. Sie ist kein beobachtetes individuelles Gehalt, sondern eine Proxy-Gehaltsklasse, die auf externer Entgeltstatistik basiert. Dadurch ist sie eng mit Merkmalen wie `beruf` und `region` verknüpft, was die hohe Modellgüte teilweise mit erklärt.

Trotzdem ist die Arbeit valide, weil genau diese Einschränkung nicht verschwiegen, sondern mit Ablation, Fehleranalyse und Datenvergleich explizit untersucht wurde. Das Projekt zeigt damit nicht nur ein funktionierendes Modell, sondern auch eine kritische Auseinandersetzung mit der Datenbasis und der Aussagekraft der Ergebnisse.

## Fazit und Ausblick

Insgesamt zeigt das Projekt, dass sich aus Merkmalen von Stellenanzeigen ein Gehaltsniveau grundsätzlich vorhersagen lässt. Für die vorliegende Arbeit erwies sich das BA-Setup trotz Proxy-Zielvariable als die tragfähigere Hauptlösung, weil es eine deutlich stabilere und vollständigere Datengrundlage bot.

Gleichzeitig hat der Adzuna-Vergleich gezeigt, dass Datenquellen mit direkteren Gehaltsfeldern methodisch attraktiver wären, in der Praxis aber nicht automatisch die bessere Basis darstellen. Damit endet das Projekt mit einer begründeten methodischen Schlussentscheidung:

- **BA** als tragender Hauptansatz
- **Adzuna** als ergänzende Machbarkeitsanalyse und als Ausblick auf eine mögliche Weiterentwicklung

Ein sinnvoller nächster Schritt wäre die Nutzung einer Datenquelle mit direkter, vollständigerer und qualitativ stabilerer Gehaltsinformation, um das Problem künftig näher an beobachteten individuellen Gehaltswerten zu modellieren.
Damit zeigt das Projekt nicht nur ein funktionierendes Modellierungssetup, sondern auch, wie wichtig die Qualität und Konstruktion der Zielvariable für die Interpretation von Machine-Learning-Ergebnissen ist.

In [48]:
final_compare = pd.DataFrame({
    "Aspekt": [
        "Verwendbare Datenmenge",
        "Zielvariable",
        "Datenqualität",
        "Nähe zum eigentlichen Gehaltsproblem",
        "Eignung als Hauptansatz"
    ],
    "BA-Setup": [
        "hoch",
        "Proxy-Gehaltsklasse",
        "stabil",
        "mittel",
        "hoch"
    ],
    "Adzuna-Setup": [
        "niedrig",
        "direktere salary_class",
        "eingeschränkt",
        "hoch",
        "mittel bis niedrig"
    ]
})

final_compare

,Aspekt,BA-Setup,Adzuna-Setup
0,Verwendbare Datenmenge,hoch,niedrig
1,Zielvariable,Proxy-Gehaltsklasse,direktere salary_class
2,Datenqualität,stabil,eingeschränkt
3,Nähe zum eigentlichen Gehaltsproblem,mittel,hoch
4,Eignung als Hauptansatz,hoch,mittel bis niedrig
